<div style="border-top:4px solid #0f766e;padding:16px">DATA WAREHOUSING WITH APACHE DORIS · OPTIONAL LAB</div>

# Optional Lab 5B: MySQL → Flink CDC → Doris

**Module 5: Batch and Continuous Data Ingestion** · [Lesson](course.md) · [Quiz](quiz05_load_methods_and_retry_safety.ipynb) · [Reading: Continuous Ingestion](optional05_continuous_ingestion.md)

A Flink job reads a MySQL table with the MySQL CDC connector, first the rows that exist (the **snapshot**), then every change from the MySQL binlog, and writes them into Doris with the Doris connector. You check the snapshot, change the source and watch inserts, updates, and deletes arrive, stop the job with a savepoint while the source keeps changing, and restore the job from the savepoint so that it catches up. Module 7 builds on the same ideas with replayed events.

Plan about 40 minutes. The Lab is optional: no other Lab depends on it.

**What you need**

- Docker with at least 10 GiB of memory and 4 CPUs (Docker Desktop: Settings → Resources). In this Lab Doris used at most 2.8 GiB, the Flink JobManager 1.8 GiB, the TaskManager 1.1 GiB, and MySQL 0.5 GiB, measured on Docker Desktop with Apple silicon. The first cell checks Docker's capacity before it starts anything.
- Downloads the first time: `mysql:8.0.33` (about 800 MB) and `flink:1.20.1-scala_2.12-java11`, plus three connector JARs from Maven Central (about 65 MB) that Compose builds into the course image `doris-course-flink` (1.5 GB) in about a minute.
- The course sandbox from Lab 1. The first cell recreates the Doris container with a 6 GiB memory limit; its data stays. Finish the other notebooks first: a main Lab's startup cell removes the limit again.

The Lab rebuilds only `dw_course_l1_streaming.ext_cdc_orders`, and it empties the course's own MySQL table `course_cdc.orders` before inserting three simulated orders; these are not WWI orders and have their own amounts. It synchronizes **one table with a fixed schema**: not a whole database, and not automatic schema changes. Use only the course's demonstration accounts. The [environment guide](../../environments/streaming/README.md) lists the services and how to stop them.

If you cannot run the Lab, read it anyway: each step's **Expected result** shows what a run prints.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import show_response, show_records
from dw_course.streaming import (
    prepare_streaming, mysql, flink_api, submit_sql, wait_checkpoint, stop_with_savepoint,
    wait_rows, check_flink_job, cleanup_cdc_jobs,
)
prepare_environment(start=True, streaming=True, streaming_profile="cdc")
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("Isolated lab database:", lab.database)

prepare_streaming("cdc", start=True)

**Expected result**

`Docker capacity: … GiB RAM, … CPUs`, at least 10 GiB and 4 CPUs, then the startup panel with six green steps (the first time, Compose recreates the Doris container with the 6 GiB limit) and `Isolated lab database: dw_course_l1_streaming`. Compose starts MySQL, the JobManager, and the TaskManager; on a first run it builds the Flink image first. `Flink Web UI: <port>` is a free port that Docker chose on 127.0.0.1: open `http://127.0.0.1:<port>` to watch the job while you work. The wait card turns green when MySQL answers and Flink has a free task slot.

## 1. Reset the source and the target

The setup cancels stale jobs named `course_mysql_orders` that an earlier run left behind; it refuses to touch any other running Flink job. Then it empties `course_cdc.orders` in MySQL and inserts three orders, and rebuilds `ext_cdc_orders` in Doris as a Merge-on-Write Unique Key table, so that each order is one row that updates and deletes can change.

MySQL writes every change to its **binlog**; the course's MySQL keeps whole rows (ROW / FULL) for seven days. The CDC user can read the table and the binlog (`REPLICATION SLAVE`, `REPLICATION CLIENT`). `SHOW MASTER STATUS` shows the binlog file and position that MySQL has written up to: the end of the log, not a position that any reader has reached.

In [ ]:
cleanup_cdc_jobs()
active = [j for j in flink_api("/jobs/overview")["jobs"] if j["state"] not in {"FINISHED","CANCELED","FAILED"}]
expect(active, [], title="No Flink job of this Lab is running")
mysql("""DELETE FROM course_cdc.orders;
INSERT INTO course_cdc.orders VALUES
(920001,1,'CREATED',100.00),(920002,2,'CREATED',200.00),(920003,3,'CREATED',50.00);
SHOW MASTER STATUS;
SELECT * FROM course_cdc.orders ORDER BY order_id;""")
lab.execute("DROP TABLE IF EXISTS ext_cdc_orders")
lab.execute("""CREATE TABLE ext_cdc_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""");

**Expected result**

A green check that no Flink job of this Lab is running, then two result tables from MySQL: `SHOW MASTER STATUS` with a `File` such as `mysql-bin.000003` and a `Position`, and the three source orders 920001, 920002, 920003, all CREATED, 350.00 in total. No error from Doris.

## 2. Submit the synchronization job

The Flink SQL defines a source table on MySQL with `'scan.startup.mode'='initial'`, snapshot first, then the binlog, and a sink table on the Doris table, then runs `INSERT INTO doris_orders SELECT … FROM mysql_orders` as a continuous job:

| Setting | Meaning |
| --- | --- |
| `'execution.checkpointing.interval' = '5s'` | Every 5 seconds the job saves a **checkpoint**: its binlog position and the state of its operators |
| `'sink.enable-2pc'='true'`, `'sink.label-prefix'` | The sink writes each checkpoint's rows as one Doris transaction and commits it when the checkpoint completes; the labels start with a prefix that is new for each experiment |
| `'sink.enable-delete'='true'` | A DELETE in MySQL deletes the row in Doris |
| `'fenodes'='doris:8030'`, `'benodes'='doris:8040'`, `'auto-redirect'='false'` | Flink reaches Doris by its container name on the course network, and loads straight into the BE: the sandbox's BE is registered as 127.0.0.1, which inside the Flink container would be Flink itself |

A job in the state RUNNING has only started: the next steps check the rows in Doris.

In [ ]:
label = "course_cdc_" + uuid4().hex[:12]
sql = f"""SET 'execution.checkpointing.interval' = '5s';
SET 'parallelism.default' = '1';
SET 'pipeline.name' = 'course_mysql_orders';
CREATE TABLE mysql_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='mysql-cdc','hostname'='mysql','port'='3306',
 'username'='course_cdc','password'='course_cdc_local_only',
 'database-name'='course_cdc','table-name'='orders','server-id'='55101-55102',
 'server-time-zone'='Asia/Shanghai','scan.startup.mode'='initial'
);
CREATE TABLE doris_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='doris','fenodes'='doris:8030','benodes'='doris:8040','auto-redirect'='false','table.identifier'='dw_course_l1_streaming.ext_cdc_orders',
 'username'='root','password'='', 'sink.label-prefix'='{label}',
 'sink.enable-delete'='true','sink.enable-2pc'='true',
 'sink.properties.format'='json','sink.properties.read_json_by_line'='true'
);
INSERT INTO doris_orders SELECT order_id,customer_id,order_status,order_amount FROM mysql_orders;
"""
print(sql)
job_id = submit_sql(sql)
print("Job ID:", job_id)

**Expected result**

The printed Flink SQL, then the SQL client's log: three `SET` statements and two `CREATE TABLE` statements succeed, the INSERT is submitted, and the log ends with `Job ID: <32 hexadecimal digits>`. The wait card turns green when the job is RUNNING, and the cell prints the job ID again. Warnings about `jdk.compiler` and a dumb terminal come from the SQL client and do not matter.

### Wait for the snapshot

The job first reads the rows that exist in MySQL. Wait until Doris holds the same three rows, then wait for the first completed checkpoint.

In [ ]:
baseline = [[920001,1,"CREATED","100.00"],[920002,2,"CREATED","200.00"],[920003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_cdc_orders",baseline, check_health=lambda: check_flink_job(job_id)),baseline,
       title="The snapshot arrived")
show_records("Completed checkpoint", [(wait_checkpoint(job_id)["latest"]["completed"],)], columns=["Completed checkpoints"])
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="Snapshot results");

**Expected result**

The wait turns green within about 10 seconds, with the three CREATED orders, 350.00 in total, like the source. `Completed checkpoints` is at least 1: a checkpoint completes every 5 seconds, and only then does the sink commit its Doris transaction.

## 3. Change the source and watch the changes arrive

Update 920001 to PAID, delete 920002, and insert 920004, all in MySQL. The delete in Doris comes from the binlog event: the notebook runs no DELETE against Doris. Expect three rows totaling 225.00.

In [ ]:
mysql("""UPDATE course_cdc.orders SET order_status='PAID' WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920002;
INSERT INTO course_cdc.orders VALUES (920004,4,'CREATED',75.00);
SELECT * FROM course_cdc.orders ORDER BY order_id;
SHOW MASTER STATUS;""")
changed = [[920001,1,"PAID","100.00"],[920003,3,"CREATED","50.00"],[920004,4,"CREATED","75.00"]]
expect(wait_rows(lab,"ext_cdc_orders",changed, check_health=lambda: check_flink_job(job_id)),changed,
       title="The update, the delete, and the insert arrived")
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"225.00"]],
       title="Three orders, 225.00")

**Expected result**

MySQL shows 920001 PAID, 920003, and 920004, and `SHOW MASTER STATUS` shows a higher `Position`. Within about 10 seconds Doris holds the same three rows, 225.00 in total: the update replaced 920001's row, and 920002 is gone.

## 4. Stop the job with a savepoint while the source keeps changing

A **savepoint** is a checkpoint that you ask for: `flink stop` writes one and stops the job. It holds the job's binlog position and the state of its operators. This is a controlled stop and restore, not a crash or an automatic failover.

After the stop, change the source three more times. `SHOW MASTER STATUS` moves on; the savepoint does not, and Doris keeps its rows from step 3. Do not delete binlog files, the MySQL volume, or the Flink state volume: the restore needs all three.

In [ ]:
savepoint = stop_with_savepoint(job_id)
print("Savepoint path:", savepoint)
mysql("""UPDATE course_cdc.orders SET order_amount=125.00 WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920003;
INSERT INTO course_cdc.orders VALUES (920005,5,'CREATED',40.00);
SHOW MASTER STATUS;""")
expect(lab.query("SELECT * FROM ext_cdc_orders ORDER BY order_id"), changed,
       title="Doris keeps the rows of step 3 while the job is stopped")

**Expected result**

`Savepoint completed. Path: file:/opt/flink/state/savepoints/savepoint-<6 hex>-<12 hex>` and `Savepoint path: …` with the same path; the wait turns green when the job is FINISHED. MySQL's `SHOW MASTER STATUS` shows a higher `Position` after the three changes, and Doris still holds the three rows of step 3.

## 5. Restore from the savepoint and catch up

Submit the same SQL with only `'execution.savepoint.path'` added: the same tables, label prefix, and parallelism. Do not reset the source or the target. The new job continues from the savepoint's binlog position, so it applies the three changes made while it was stopped. Expect three rows totaling 240.00, and check in Flink's record of the restore that it started from the savepoint.

In [ ]:
restored_job = submit_sql(f"SET 'execution.savepoint.path' = '{savepoint}';\n" + sql)
final = [[920001,1,"PAID","125.00"],[920004,4,"CREATED","75.00"],[920005,5,"CREATED","40.00"]]
expect(wait_rows(lab,"ext_cdc_orders",final, check_health=lambda: check_flink_job(restored_job)),final,
       title="The job caught up after the restore")
checkpoints = wait_checkpoint(restored_job)
restored = checkpoints["latest"]["restored"]
expect(restored["external_path"],savepoint, title="The job restored from the savepoint")
show_response(restored, title="Restoration record")
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"240.00"]],
       title="Three orders, 240.00")
mysql("SELECT * FROM course_cdc.orders ORDER BY order_id")
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="Results after restoration");

**Expected result**

A new job ID, then the three rows 920001 PAID 125.00, 920004 CREATED 75.00, and 920005 CREATED 40.00: 240.00 in total, with 920003 deleted. The restoration record has `is_savepoint` True and `external_path` equal to the savepoint path. The two MySQL result sets match Doris.

## 6. Practice: restore from a newer savepoint

Stop the restored job with a new savepoint, change order 920004 to PAID in MySQL while it is stopped, restore from the new savepoint, and check that Doris catches up and that Flink restored from the new path. Keep the new job's ID in `restored_job`: the last cell stops whichever job that variable names.

Then explain why the time of the latest change in MySQL could not replace the savepoint.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
practice_savepoint = stop_with_savepoint(restored_job)
mysql("UPDATE course_cdc.orders SET order_status='PAID' WHERE order_id=920004;")
practice_job = submit_sql(f"SET 'execution.savepoint.path' = '{practice_savepoint}';\n" + sql)
practice_rows = [[920001,1,"PAID","125.00"],[920004,4,"PAID","75.00"],[920005,5,"CREATED","40.00"]]
expect(wait_rows(lab, "ext_cdc_orders", practice_rows, check_health=lambda: check_flink_job(practice_job)),
       practice_rows, title="Doris caught up with the change made while the job was stopped")
expect(wait_checkpoint(practice_job)["latest"]["restored"]["external_path"], practice_savepoint,
       title="The job restored from the new savepoint")
restored_job = practice_job  # the last cell stops this job

**Expected result**

A savepoint path, a new job ID, and Doris holding 920001 PAID 125.00, 920004 PAID 75.00, and 920005 CREATED 40.00; the restored checkpoint is the new savepoint.

A business time, such as the time an order last changed, cannot replace the savepoint. The savepoint records the binlog file and position that the job had read, and the state of its operators, including the sink's pending Doris transactions. A time written by the application does not identify a position: several changes can share a second, a deleted row leaves no time to read, and a transaction that committed late can carry an earlier time. Restarting from such a time skips changes or applies them twice.

## 7. Stop the job and keep the results

Stop the running job with a final savepoint and close the connection. Doris keeps its rows, and the savepoints stay in the Flink state volume, so you can restore this job later with the printed path. This Lab checks normal operation and a controlled restore; it does not show exactly-once behavior for every failure, events that arrive out of order, visibility across several tables, or production load.

In [ ]:
final_savepoint = stop_with_savepoint(restored_job)
print("Final savepoint path:",final_savepoint)
lab.close()

**Expected result**

`Final savepoint path: file:/opt/flink/state/savepoints/savepoint-…`. No Flink job runs any more; MySQL and Flink keep running until you stop them.

## You are done when

- you have seen the snapshot, an update, a delete, and an insert from MySQL arrive in Doris;
- you can explain what a checkpoint and a savepoint hold, and why `SHOW MASTER STATUS` is not the job's position;
- you have restored the job from a savepoint and checked that it caught up with the changes made while it was stopped.

To stop MySQL and Flink and keep their data, run the stop command in the [environment guide](../../environments/streaming/README.md#troubleshooting-and-shutdown).

## Next

Take the [continuous ingestion quiz](quiz05_continuous_ingestion.ipynb) if you have not, or return to the [Module 5 lesson](course.md). Module 6 starts with the [Module 6 lesson](../module06-data-quality/course.md); Module 7 replays order events after an interruption.